# 03 — Create Multilingual Dataset

This notebook runs the code-mixed generation and multilingual dataset assembly.

**Prerequisites:** Run notebooks 01 and 02 first.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir('/content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil')
print('Working directory:', os.getcwd())

Working directory: /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil


In [4]:
# Step 1: Generate code-mixed tweets
!python scripts/create_codemixed.py

Loading English dataset...
Loaded 14427 English tweets
Loading Tamil dataset...
Loaded 14427 Tamil tweets
Found 14427 EN-TA pairs
/content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/scripts/create_codemixed.py:236: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = pairs.groupby("sentiment", group_keys=False).apply(
Selected 7213 tweets for code-mixing (50% of pairs)
Generated 6940 code-mixed tweets (skipped 273 too short or single-language)

--- Code-Mixed Dataset Statistics ---
Total: 6940

By sentiment:
    negative: 4488
     neutral: 1417
    positive: 1035

By source:
   synthetic_codemixed: 6940

By split:
         train: 4913
          test: 1029
    validation: 998

--- Sampl

In [5]:
# Step 2: Assemble unified multilingual dataset
!python scripts/create_multilingual_dataset.py

Loaded English: 14427 records
Loaded Tamil: 14427 records
Loaded Code-Mixed: 6940 records

Combined dataset: 35794 records

--- Data Leakage Check ---
  [OK] No data leakage detected!
     All 14427 original tweets have consistent splits across language variants

--- Language Balancing ---
Before balancing:
     en:  14427 (40.3%)
     ta:  14427 (40.3%)
  en-ta:   6940 (19.4%)

Target per language (train): {'en': 8537, 'ta': 8285, 'en-ta': 8285}
/content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil/scripts/create_multilingual_dataset.py:125: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = lang_df.groupby("sentiment", group_keys=False).apply(
/content/drive/MyDrive/Airline_BERT_Multil

In [6]:
# Verify outputs
import pandas as pd

for split in ['train', 'validation', 'test']:
    df = pd.read_csv(f'data/{split}.csv')
    print(f'{split}: {len(df)} rows')
    print(df['language'].value_counts())
    print()

train: 25105 rows
language
en       8536
en-ta    8285
ta       8284
Name: count, dtype: int64

validation: 5326 rows
language
en       2164
ta       2164
en-ta     998
Name: count, dtype: int64

test: 5359 rows
language
en       2165
ta       2165
en-ta    1029
Name: count, dtype: int64



In [8]:
# Data leakage check
multi_df = pd.read_csv('data/multilingual_dataset.csv')
id_splits = multi_df.groupby('original_tweet_id')['split'].nunique()
leaky = id_splits[id_splits > 1]
print(f'Leaky IDs: {len(leaky)}') #  (should be 0)

Leaky IDs: 0
